In [1]:
import jax.numpy as jnp
import jax
import numpy as np
import plotly.graph_objects as go
import jax.scipy.sparse.linalg
import timeit
from splank import SoftPlankton, Sphere

## Icosahedron

In [2]:
def icosahedron():
    """Generate a unit icosahedron: 12 vertices, 20 triangular faces."""
    t = (1 + np.sqrt(5)) / 2
    vertices = np.array([
        [-1,  t,  0], 
        [ 1,  t,  0], 
        [-1, -t,  0], 
        [ 1, -t,  0],
        [ 0, -1,  t], 
        [ 0,  1,  t], 
        [ 0, -1, -t], 
        [ 0,  1, -t],
        [ t,  0, -1], 
        [ t,  0,  1], 
        [-t,  0, -1], 
        [-t,  0,  1]
    ])
    vertices /= np.linalg.norm(vertices[0])  # Normalize to unit sphere
    faces = np.array([
        [0, 11, 5], 
        [0, 5, 1], 
        [0, 1, 7], 
        [0, 7, 10], 
        [0, 10, 11],
        [1, 5, 9], 
        [5, 11, 4], 
        [11, 10, 2], 
        [10, 7, 6], 
        [7, 1, 8],
        [3, 9, 4], 
        [3, 4, 2], 
        [3, 2, 6], 
        [3, 6, 8], 
        [3, 8, 9],
        [4, 9, 5], 
        [2, 4, 11], 
        [6, 2, 10], 
        [8, 6, 7], 
        [9, 8, 1]
    ])
    return vertices, faces

In [3]:
def midpoint(v1, v2, vertices, midpoint_cache):
    """Find midpoint of two vertices, normalize to sphere, and store index."""
    key = tuple(sorted((v1, v2)))  # Unique key for edge
    if key in midpoint_cache:
        return midpoint_cache[key]
    
    mid = (vertices[v1] + vertices[v2]) / 2
    mid /= np.linalg.norm(mid)  # Normalize to sphere
    index = len(vertices)
    vertices.append(mid)
    midpoint_cache[key] = index
    return index


In [4]:
def subdivide(vertices, faces):
    """Subdivide each triangular face into four triangles."""
    vertices = list(vertices)  # Convert to mutable list
    new_faces = []
    midpoint_cache = {}

    for tri in faces:
        a = midpoint(tri[0], tri[1], vertices, midpoint_cache)
        b = midpoint(tri[1], tri[2], vertices, midpoint_cache)
        c = midpoint(tri[2], tri[0], vertices, midpoint_cache)

        new_faces.append([tri[0], a, c])
        new_faces.append([tri[1], b, a])
        new_faces.append([tri[2], c, b])
        new_faces.append([a, b, c])

    return np.array(vertices), np.array(new_faces)

In [5]:
def icosphere(subdivisions=0):
    """Generate an icosphere by subdividing an icosahedron."""
    vertices, faces = icosahedron()
    for _ in range(subdivisions):
        vertices, faces = subdivide(vertices, faces)
    return np.array(vertices), np.array(faces)


In [6]:
def create_sphere(center, radius, color='r', opacity=0.5, resolution=10):
    """Generates a transparent sphere for Plotly."""
    u, v = np.mgrid[0:np.pi:resolution*1j, 0:2*np.pi:resolution*1j]
    x = radius * np.sin(u) * np.cos(v) + center[0]
    y = radius * np.sin(u) * np.sin(v) + center[1]
    z = radius * np.cos(u) + center[2]

    return go.Surface(x=x, y=y, z=z, colorscale=[[0, color], [1, color]], opacity=opacity, showscale=False)


In [7]:
# Example of visu
num_subdivisions = 2
vertices, faces = icosphere(num_subdivisions)
spheres = [create_sphere([0, 0, 0], radius=1, color='cyan', opacity = 0.7, resolution=100)]

# Plot spheres on vertices
for v in vertices:
    spheres.append(create_sphere(v, radius=0.1, color='red', opacity=1))

# Set limits and view
fig = go.Figure(data=spheres)
fig.update_layout(scene=dict(aspectmode="cube"))

# Customize layout
fig.update_layout(
    title="Icosahdron with " + str(len(vertices)) + " vertices" ,
    width=900, height=700,  # Adjust figure size
    scene=dict(
        xaxis_title="X Label",  
        yaxis_title="Y Label",  
        zaxis_title="Z Label",
        xaxis=dict(visible=False),  # Hide X axis
        yaxis=dict(visible=False),  # Hide Y axis
        zaxis=dict(visible=False),  # Hide Z axis
        aspectmode="cube",  # Keep aspect ratio cube-like
    ),
    margin=dict(l=0, r=0, t=40, b=0)  # Reduce margins
)

fig.show()

## Inverse methods for a positive definite symmetric matrix 

In [8]:
# Inverse using Cholesky decomposition
def cholesky_inverse(M):
    L = jnp.linalg.cholesky(M)
    L_inv = jax.scipy.linalg.solve_triangular(L, jnp.eye(L.shape[0]), lower=True)
    return L_inv.T @ L_inv

# Inverse using solve M.X = I
def solve_inverse(M):
    I = jnp.eye(M.shape[0])
    Minv = jnp.linalg.solve(M, I)  # Solves M @ X = I
    return Minv

# Preconditioned method
def preconditioned_inverse(M):
    D = jnp.diag(M)
    D_inv_sqrt = jnp.diag(1.0 / jnp.sqrt(D + 1e-10))
    M_tilde = D_inv_sqrt @ M @ D_inv_sqrt
    I = jnp.eye(M.shape[0])
    Mred_tilde = jnp.linalg.solve(M_tilde, I)
    return D_inv_sqrt @ Mred_tilde @ D_inv_sqrt

# Iterative CG-based inverse
def cg_inverse(M, maxiter=1000, tol=1e-6):
    I = jnp.eye(M.shape[0])
    return jax.vmap(
        lambda b: jax.scipy.sparse.linalg.cg(M, b, maxiter=maxiter, tol=tol)[0],
        in_axes=1,
        out_axes=1,
    )(I)

# Preconditioned CG (Jacobi) inverse
def pcg_inverse(M, maxiter=1000, tol=1e-6):
    D = jnp.diag(M)
    D_inv = 1.0 / (D + 1e-10)
    preconditioner = lambda x: D_inv * x
    I = jnp.eye(M.shape[0])
    
    def solve_column(b):
        return jax.scipy.sparse.linalg.cg(
            M, b, M=preconditioner, maxiter=maxiter, tol=tol
        )[0]
    
    return jax.vmap(solve_column, in_axes=1, out_axes=1)(I)


## Motility

In [9]:
# Verification and timing
def test_methods(M, trials=10):
    # JIT compile all methods
    methods = {
        "Cholesky": jax.jit(cholesky_inverse),
        "Solve": jax.jit(solve_inverse),
        "Preconditioned": jax.jit(preconditioned_inverse),
        "JAX Default": jax.jit(jnp.linalg.inv),
        "CG": jax.jit(cg_inverse),
        "PCG": jax.jit(pcg_inverse),
    }

    # Verify accuracy
    ref_inv = methods["JAX Default"](M)
    print("Validation results:")
    for name, fn in methods.items():
        inv = fn(M).block_until_ready()
        error = jnp.linalg.norm(inv - ref_inv)
        print(f"{name:12} | Error: {error:.2e}")

    # Time methods
    print("\nTiming results:")
    for name, fn in methods.items():
        fn(M).block_until_ready()  # Warmup
        time = timeit.timeit(lambda: fn(M).block_until_ready(), number=trials) / trials
        print(f"{name:12} | {time:.5f} s")

In [13]:
num_subdivisions = 1
mysphere = SoftPlankton()

vertices, _ = icosphere(num_subdivisions)
for vertex in vertices:
    mysphere.add_sphere(Sphere(position=vertex, radius=0.25))

M = mysphere.compute_mobility_tensor()
print("Nb of spheres:", len(vertices), "  M shape:", M.shape)

# Run test with matrix M
test_methods(M, trials=1)

Nb of spheres: 42   M shape: (252, 252)
Validation results:
Cholesky     | Error: 6.60e-05
Solve        | Error: 0.00e+00
Preconditioned | Error: 1.23e-04
JAX Default  | Error: 0.00e+00
CG           | Error: 1.36e-04
PCG          | Error: 1.23e-04

Timing results:
Cholesky     | 0.00081 s
Solve        | 0.00085 s
Preconditioned | 0.00181 s
JAX Default  | 0.00076 s
CG           | 0.02865 s
PCG          | 0.06470 s
